# 4. Vanna Training

This notebook trains Vanna with the schema and sample queries for the EMR database.
It dynamically loads training data from version-controlled files under the `vanna_training` folder.

In [1]:
import os
import sys
import yaml

# Ultimate path injection to find 'src' regardless of current working directory
cwd = os.path.abspath(os.getcwd())
project_root = None

# Walk up from current working directory
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent

if project_root:
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    path_prefix = os.path.relpath(project_root, cwd)
    print(f"Project root found: {project_root}")
else:
    print("Warning: Could not automatically detect project root. Using default path prefix '..'")
    path_prefix = ".."
    project_root = os.path.abspath(path_prefix)

from src.services.providers import get_vanna

In [2]:
print("1. Initializing Vanna...")
vn = get_vanna()

In [3]:
print("2. Training Vanna with DDL...")
schema_path = os.path.join(project_root, "vanna_training", "schema.sql")
print(f"Loading schema from: {schema_path}")
with open(schema_path, "r", encoding="utf-8") as f:
    ddl = f.read()
vn.train(ddl=ddl)

In [4]:
print("3. Training Vanna with Documentation...")
doc_path = os.path.join(project_root, "vanna_training", "domain_docs.md")
print(f"Loading documentation from: {doc_path}")
with open(doc_path, "r", encoding="utf-8") as f:
    doc = f.read()
vn.train(documentation=doc)

In [5]:
print("4. Training Vanna with Sample Q&A...")
qa_path = os.path.join(project_root, "vanna_training", "qa_pairs.yaml")
print(f"Loading QA pairs from: {qa_path}")
with open(qa_path, "r", encoding="utf-8") as f:
    qa_data = yaml.safe_load(f)

for item in qa_data:
    q = item["question"]
    sql = item["sql"]
    vn.train(question=q, sql=sql)

print("Training complete.")

In [6]:
print("5. Testing SQL Generation...")
test_query = "Hitung berapa banyak kasus kerusakan yang berkaitan dengan kluster Sistem Pendingin (Cooling System) menurut GraphRAG"
sql = vn.generate_sql(test_query)
print(f"Question: {test_query}\nGenerated SQL: {sql}")

if sql:
    df = vn.run_sql(sql)
    print(df)